Instalacija svih biblioteka

In [ ]:
!git clone https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
!pip install -e ".[torch,metrics]" -q
!pip install bitsandbytes -q
!pip install unsloth -q
! pip install colab-xterm
# Force-install compatible versions
!pip install "datasets>=3.4.1,<4.0.0" "transformers>=4.55.0,<=5.2.0" -U -q

# Verify the installation for LLaMA-Factory
!pip install -e . -q


Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 26800, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 26800 (delta 8), reused 5 (delta 5), pack-reused 26780 (from 2)
Receiving objects: 100% (26800/26800), 12.85 MiB | 7.49 MiB/s, done.
Resolving deltas: 100% (19261/19261), done.
/content/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

Registrovanje dataset-a

In [ ]:
import json, shutil

# Upload output.jsonl na Colab sidebar-u, pa pokreni:
shutil.copy("/content/output.jsonl", "/content/LLaMA-Factory/data/commit_messages.jsonl")

# Prijavi dataset
with open("/content/LLaMA-Factory/data/dataset_info.json", "r+") as f:
    info = json.load(f)
    info["commit_messages"] = {
        "file_name": "commit_messages.jsonl"
    }
    f.seek(0)
    json.dump(info, f, indent=2)
    f.truncate()

print("Dataset registered!")

Dataset registered!


Prijava na huggingface

In [ ]:
from huggingface_hub import login
login()

Fine-tuning modela koristeci llama-factory-cli

In [ ]:
!llamafactory-cli train \
  --model_name_or_path meta-llama/Llama-3.2-3B-Instruct \
  --do_train true \
  --dataset commit_messages \
  --template llama3 \
  --finetuning_type lora \
  --quantization_bit 4 \
  --quantization_method bnb \
  --lora_rank 16 \
  --lora_target q_proj,k_proj,v_proj,o_proj \
  --output_dir /content/commit-lora-llama \
  --num_train_epochs 3 \
  --per_device_train_batch_size 2 \
  --gradient_accumulation_steps 4 \
  --learning_rate 2e-4 \
  --warmup_steps 50 \
  --lr_scheduler_type cosine \
  --cutoff_len 2048 \
  --fp16 true \
  --upcast_layernorm true \
  --logging_steps 10 \
  --save_steps 200 \
  --trust_remote_code true

[INFO|2026-03-28 11:44:46] llamafactory.hparams.parser:505 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.float16
[INFO|configuration_utils.py:670] 2026-03-28 11:44:46,447 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--meta-llama--Llama-3.2-3B-Instruct/snapshots/0cb88a4f764b7a12671c53f0838cd831a0843b95/config.json
[INFO|configuration_utils.py:742] 2026-03-28 11:44:46,450 >> Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 3072,
  "initializer_range": 0.02,
  "intermediate_size": 8192,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 24,
  "num_hidden_layers": 28,
  "num_key_value_hea

Patch za phi3 model - nepotrebno za llama3 model

In [ ]:
import transformers.cache_utils

# Popravljamo DynamicCache da bi Phi-3 mogao da radi na novim verzijama
def patch_phi3():
    if not hasattr(transformers.cache_utils.DynamicCache, "seen_tokens"):
        def get_seen_tokens(self):
            return self.get_seq_length()

        transformers.cache_utils.DynamicCache.seen_tokens = property(get_seen_tokens)
        print("✅ Uspešno patch-ovan DynamicCache: 'seen_tokens' je sada dostupan.")
    else:
        print("ℹ️ 'seen_tokens' već postoji, patch nije potreban.")

patch_phi3()

Rucno testiranje istreniranog modela

In [ ]:
!llamafactory-cli chat \
  --model_name_or_path meta-llama/Llama-3.2-3B-Instruct \
  --adapter_name_or_path /content/commit-lora-llama \
  --template llama3 \
  --finetuning_type lora \
  --quantization_bit 4 \
  --quantization_method bnb

[INFO|configuration_utils.py:670] 2026-03-28 13:11:11,634 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--meta-llama--Llama-3.2-3B-Instruct/snapshots/0cb88a4f764b7a12671c53f0838cd831a0843b95/config.json
[INFO|configuration_utils.py:742] 2026-03-28 13:11:11,637 >> Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "dtype": "bfloat16",
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 3072,
  "initializer_range": 0.02,
  "intermediate_size": 8192,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 24,
  "num_hidden_layers": 28,
  "num_key_value_heads": 8,
  "pad_token_id": null,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "factor": 32.0,
    "high_freq_factor": 4.0,
    "low_freq_fa

KeyboardInterrupt: 

Exportovanje i mergeovanje modela i LoRa adaptera

In [ ]:
!llamafactory-cli export \
  --model_name_or_path meta-llama/Llama-3.2-3B-Instruct \
  --adapter_name_or_path /content/commit-lora-llama \
  --template llama3 \
  --finetuning_type lora \
  --export_dir /content/commit-model-merged \
  --export_size 2 \
  --export_device cpu \
  --trust_remote_code true

/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")
config.json: 100% 878/878 [00:00<00:00, 2.79MB/s]
[INFO|configuration_utils.py:670] 2026-03-28 18:29:53,828 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--meta-llama--Llama-3.2-3B-Instruct/snapshots/0cb88a4f764b7a12671c53f0838cd831a0843b95/config.json
[INFO|configuration_utils.py:742] 2026-03-28 18:29:53,831 >> Model config LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "b

Povezivanje na google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Cuvanje merge-ovanog modela na drive

In [ ]:
import shutil
shutil.copytree('/content/commit-model-merged', '/content/drive/MyDrive/mergedModel/commit-model-merged',dirs_exist_ok=True)
print("Model sacuvan!")

Model sacuvan!


Cuvanje LoRa adaptera

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copytree('/content/commit-lora-llama', '/content/drive/MyDrive/mergedModel/commit-lora-llama')
print("Sacuvano!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Sacuvano!


Ucitavanje modela sa drive-a

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil

# shutil.copytree('/content/drive/MyDrive/commit-lora-llama','/content/commit-lora-llama')
shutil.copytree('/content/drive/MyDrive/mergedModel/commit-model-merged','/content/commit-model-merged')

print('Ucitano')

Mounted at /content/drive
Ucitano


In [ ]:
shutil.copytree('/content/drive/MyDrive/commit-model-merged','/content/commit-model-merged')

'/content/commit-model-merged'

Konvertuj model u GGUF i kopiraj ga na drive

In [ ]:
# Instaliraj llama.cpp conversion alat
!pip install llama-cpp-python -q
!git clone https://github.com/ggerganov/llama.cpp.git
!pip install -r llama.cpp/requirements.txt -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.0/59.0 MB 14.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.4 MB/s eta 0:00:00
Cloning into 'llama.cpp'...
remote: Enumerating objects: 85791, done.
remote: Counting objects: 100% (101/101), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 85791 (delta 54), reused 27 (delta 27), pack-reused 85690 (from 3)
Receiving objects: 100% (85791/85791), 349.10 MiB | 47.22 MiB/s, done.
Resolving deltas: 100% (61923/61923), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 59.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 50

In [ ]:
from huggingface_hub import hf_hub_download
import shutil
import os

dst = '/content/commit-model-merged'
os.makedirs(dst, exist_ok=True)

for f in ['tokenizer.json', 'tokenizer_config.json', 'special_tokens_map.json']:
    path = hf_hub_download(
        repo_id="meta-llama/Llama-3.2-3B-Instruct",
        filename=f
    )
    shutil.copy(path, f'{dst}/{f}')
    print(f'Preuzet: {f}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

Preuzet: tokenizer.json


tokenizer_config.json: 0.00B [00:00, ?B/s]

Preuzet: tokenizer_config.json


special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Preuzet: special_tokens_map.json


In [ ]:
!python /content/llama.cpp/convert_hf_to_gguf.py \
  /content/commit-model-merged \
  --outfile /content/commit-model-f16.gguf \
  --outtype f16

INFO:hf-to-gguf:Loading model: commit-model-merged
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00004.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00004.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00003-of-00004.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00004-of-00004.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_freqs.weight,           torch.float32 --> F32, shape = {64}
INFO:hf-to-gguf:token_embd.weight,           torch.bfloat16 --> F16, shape = {3072, 128256}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.bfloat16 --> F32, shape = {3072}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.bfloat16 --> F16, shape = {8192, 3072}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.bfl

In [ ]:
# Build llama-quantize
!cmake /content/llama.cpp -B /content/llama.cpp/build -DCMAKE_BUILD_TYPE=Release
!cmake --build /content/llama.cpp/build --config Release -j4

# Kvantizuj
!/content/llama.cpp/build/bin/llama-quantize \
  /content/commit-model-f16.gguf \
  /content/commit-model-q4.gguf \
  Q4_K_M

-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- Found OpenMP_C: 

In [ ]:
# Sačuvaj na Drive
import shutil
shutil.copy('/content/commit-model-q4.gguf', '/content/drive/MyDrive/mergedModel/commit-model-q4.gguf')
print("GGUF sacuvan!")

GGUF sacuvan!
